In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!rm -rf ./results submission.csv

In [3]:
# train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [4]:
# submission_df=pd.DataFrame({
#     'ID':test_data['id'],
#     'Prediction':'ABC'
# })

In [5]:
# submission_df.to_csv('submission.csv',index=False)

In [6]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [7]:
# RESET AND RUN DEBERTA WITH CLASS WEIGHTS DIRECTLY
import os, gc
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForMultipleChoice,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from sklearn.model_selection import StratifiedShuffleSplit
from dataclasses import dataclass
from collections import Counter

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.manual_seed(42); np.random.seed(42)

TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH  = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 128
BATCH_SIZE = 2
GRAD_ACCUM = 8
SEED = 42

option_to_index = {'A':0,'B':1,'C':2,'D':3,'E':4}
index_to_option = {0:'A',1:'B',2:'C',3:'D',4:'E'}

train_df = pd.read_csv(TRAIN_PATH).fillna("")
test_df  = pd.read_csv(TEST_PATH).fillna("")
train_df['label'] = train_df['answer'].map(option_to_index)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
tr_idx, val_idx = next(sss.split(train_df, train_df['label']))
df_train = train_df.iloc[tr_idx].reset_index(drop=True)
df_val   = train_df.iloc[val_idx].reset_index(drop=True)
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def preprocess_function(examples):
    n = len(examples['prompt'])
    first_s, second_s = [], []
    for i in range(n):
        p = str(examples['prompt'][i])
        for opt in ['A','B','C','D','E']:
            first_s.append(p)
            second_s.append(str(examples[opt][i]))
    tok = tokenizer(first_s, second_s, truncation=True,
                    max_length=MAX_LENGTH, padding='max_length',
                    return_tensors=None)
    return {k:[v[i*5:(i+1)*5] for i in range(n)] for k,v in tok.items()}

def make_ds(df):
    drop = [c for c in df.columns if c not in
            ['input_ids','attention_mask','token_type_ids','label']]
    return Dataset.from_pandas(df).map(
        preprocess_function, batched=True,
        batch_size=100, remove_columns=drop)

def make_test_ds(df):
    drop = [c for c in df.columns if c not in
            ['input_ids','attention_mask','token_type_ids']]
    return Dataset.from_pandas(df).map(
        preprocess_function, batched=True,
        batch_size=100, remove_columns=drop)

print("Tokenizing...")
tok_train = make_ds(df_train)
tok_val   = make_ds(df_val)
tok_test  = make_test_ds(test_df)

s = tok_train[0]
assert len(s['input_ids'])==5 and len(s['input_ids'][0])==MAX_LENGTH
print("✅ Tokenization correct!")

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    def __call__(self, features):
        has_labels = 'label' in features[0]
        labels = [int(f.pop('label')) for f in features] if has_labels else None
        bs = len(features); nc = len(features[0]['input_ids'])
        flat = [{k:v[i] for k,v in f.items()} for f in features for i in range(nc)]
        batch = self.tokenizer.pad(flat, padding=True, return_tensors='pt')
        batch = {k:v.view(bs,nc,-1) for k,v in batch.items()}
        if labels is not None:
            batch['labels'] = torch.tensor(labels, dtype=torch.long)
        return batch

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    top3 = np.argsort(-logits, axis=1)[:,:3]
    scores = []
    for i, true in enumerate(labels):
        if   true==top3[i,0]: scores.append(1.0)
        elif true==top3[i,1]: scores.append(0.5)
        elif true==top3[i,2]: scores.append(1/3)
        else:                 scores.append(0.0)
    return {'map_at_3': np.mean(scores)}

# Class weights to fix B>C>A>D>E imbalance
label_counts = Counter(df_train['label'].values)
total = len(df_train)
class_weights = torch.tensor(
    [total/(5*label_counts.get(i,1)) for i in range(5)],
    dtype=torch.float)
print("Class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

!rm -rf ./results
gc.collect(); torch.cuda.empty_cache()

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model = model.float()

args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch', save_strategy='epoch',
    learning_rate=5e-6,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=8,
    weight_decay=0.01, warmup_steps=100,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='map_at_3',
    greater_is_better=True,
    report_to='none',
    fp16=False, bf16=False,
    max_grad_norm=1.0, dataloader_num_workers=0,
    gradient_checkpointing=True,
    seed=SEED, logging_steps=50, save_total_limit=1,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=tok_train, eval_dataset=tok_val,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

gc.collect(); torch.cuda.empty_cache()
print("Training with class-weighted loss...")
trainer.train()

print("\n=== History ===")
for log in trainer.state.log_history:
    if 'eval_map_at_3' in log:
        print(f"Epoch {float(log['epoch']):.0f} | MAP@3: {log['eval_map_at_3']:.4f}")

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x); return e/e.sum(axis=1, keepdims=True)

raw = trainer.predict(tok_test).predictions
preds = [' '.join([index_to_option[j] for j in r[:3]])
         for r in np.argsort(-softmax(raw), axis=1)]

pd.DataFrame({'id':test_df['id'],'Prediction':preds}).to_csv(
    '/kaggle/working/submission.csv', index=False)
print("✅ Done! Submit /kaggle/working/submission.csv")

Train: 1800 | Val: 200


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizing...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


=== History ===
Epoch 1 | MAP@3: 0.4317
Epoch 2 | MAP@3: 0.5592
Epoch 3 | MAP@3: 0.7158
Epoch 4 | MAP@3: 0.7608
Epoch 5 | MAP@3: 0.8400
Epoch 6 | MAP@3: 0.8392
Epoch 7 | MAP@3: 0.8483
Epoch 8 | MAP@3: 0.8517


✅ Done! Submit /kaggle/working/submission.csv
